# Nedbrydning af variabiliteten i skadesafregning på tværs af et forsikringsselskabs organisatoriske hierarki med PROC NESTED


## Resumé

Et skadesforsikringsselskab vil vide, **hvor** uensartetheden i
skadesafregninger opstår: skyldes den forskelle mellem geografiske
regioner, mellem filialer, mellem individuelle sagsbehandlere, eller
er det blot claim-til-claim-støj? Denne notebook bygger et syntetisk,
fuldt indlejret datasæt over bilskader (Region > Filial > Sagsbehandler
> skadessag) og bruger **PROC NESTED** til at udføre en
tilfældige-effekter-variansanalyse, der estimerer variankomponenten
bidraget af hvert niveau i hierarkiet og rapporterer hver som en
procent af totalen. Resultatet fortæller ledelsen, hvilket
organisatorisk lag der skal målrettes for at gøre afregningerne mere
ensartede.



## Datakilder

Alle data genereres direkte i det første DATA step — ingen eksterne
filer, ingen netværksadgang. Designet er **fuldstændigt indlejret**:
hver filial hører til præcis én region, hver sagsbehandler til præcis
én filial, og hver skadessag til præcis én sagsbehandler.

| Datasæt | Rækker | Variabel | Type | Rolle | Beskrivelse |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (niveau 1, yderst) | Geografisk region (5 regioner: HO, SJ, SY, MJ, NJ) |
| | | `Branch` | char | CLASS (niveau 2, indlejret i Region) | Filialkode (2 filialer pr. region) |
| | | `Adjuster` | char | CLASS (niveau 3, indlejret i Branch) | Sagsbehandler-ID (2 sagsbehandlere pr. filial) |
| | | `Settlement` | num | VAR (afhængig) | Erstatning udbetalt på bilskaden, i USD |
| | | `CycleDays` | num | VAR (afhængig) | Dage fra første skadesanmeldelse til afregning |

Syntetisk struktur: 5 regioner x 2 filialer x 2 sagsbehandlere x 2
skadessager = 40 observationer. En tilfældig regionseffekt, en
tilfældig filial-inden-for-region-effekt, en tilfældig
sagsbehandler-inden-for-filial-effekt og et residual på skadessagsniveau
lægges additivt sammen via `rand('NORMAL', ...)`, så variankomponenterne
kan genfindes. Regionseffekten trækkes med den største
standardafvigelse (2.200), så *hvor* en skade behandles, er den
dominerende driver. Seed fastsat med `call streaminit(20260531)`. Et
kompakt, fuldt balanceret design holder hvert niveau af hierarkiet
befolket med reelle frihedsgrader.



# Nedbrydning af variabiliteten i skadesafregning med PROC NESTED

Når et skadesforsikringsselskab gennemgår, hvor meget det betaler for
at afregne bilskader, finder det ofte stor variation. Noget af den
variation er uundgåelig (enhver ulykke er forskellig), men noget af
den afspejler **organisatoriske** faktorer — én region afregner rigere
end en anden, en filial er mere gavmild, en individuel sagsbehandler
er en outlier.

Dataene har en strengt **indlejret** (hierarkisk) struktur:

```
Region  ->  Filial (indlejret i Region)  ->  Sagsbehandler (indlejret i Filial)  ->  individuelle skadessager
```

En filial hører til præcis én region, og en sagsbehandler til præcis
én filial, så faktorerne er indlejrede snarere end krydsede.
`PROC NESTED` udfører en tilfældige-effekter-variansanalyse for
netop dette design og estimerer en **variankomponent** på hvert
niveau, udtrykt som en procent af totalen. Den procentvise fordeling
besvarer forretningsspørgsmålet: *hvilket lag af organisationen bør vi
målrette for at gøre afregningerne mere ensartede?*



## Trin 1 — Generér et syntetisk, fuldt indlejret datasæt over skadessager

Vi simulerer 5 regioner, hver med 2 filialer, hver med 2
sagsbehandlere, hver med 2 skadessager (40 rækker i alt). Vi opbygger
den afhængige variabel af additive tilfældige effekter, så hvert
niveau reelt bidrager med varians:

- en **region**-effekt (størst spredning — regioner adskiller sig mest),
- en **filial-inden-for-region**-effekt,
- en **sagsbehandler-inden-for-filial**-effekt,
- og et **residual på skadessagsniveau** (støjen inden for en
  sagsbehandler).

`call streaminit` fastsætter seed for reproducerbarhed; hver effekt
trækkes med `rand('NORMAL', mean, sd)`. Regionseffekten bruger den
bredeste standardafvigelse, så den bør stå for den største andel af
den samlede varians. Vi genererer også en anden afhængig variabel,
`CycleDays`, der deler samme hierarki, så vi senere kan demonstrere
analysen med flere afhængige variable.



In [1]:
data claims;
   CALL streaminit(20260531);
   LÆNGDE Region $2 Branch $6 Adjuster $9;

   /* Regionsniveau tilfældig effekt: regioner adskiller sig mest. */
   GØR r = 1 TIL 5;
      Region = SCAN('HO SJ SY MJ NJ', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Filial indlejret i region. */
      GØR b = 1 TIL 2;
         Branch = catx('-', Region, SKRIV_V(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Sagsbehandler indlejret i filial. */
         GØR a = 1 TIL 2;
            Adjuster = catx('-', Branch, SKRIV_V(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Individuelle skadessager håndteret af denne sagsbehandler. */
            GØR claim = 1 TIL 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               HVIS Settlement < 0 SÅ Settlement = 0;
               HVIS CycleDays  < 1 SÅ CycleDays  = 1;
               UDDATA;
            SLUT;
         SLUT;
      SLUT;
   SLUT;

   BEHOLD Region Branch Adjuster Settlement CycleDays;
KØR;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Trin 2 — Sortér efter indlejringshierarkiet

`PROC NESTED` kræver, at inputdata er **sorteret efter
CLASS-variablene i den rækkefølge, de angives** — yderste faktor
først. Vi sorterer efter `Region Branch Adjuster`, så proceduren kan
gennemgå hierarkiet korrekt.



In [2]:
PROCEDURE SORTER data=claims;
   EFTER Region Branch Adjuster;
KØR;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Trin 3 — Variankomponentanalyse af erstatningsbeløbet

Kerneanalysen. Vi angiver CLASS-variablene yderst-til-inderst
(`Region Branch Adjuster`); den inderste replikation —
de individuelle skadessager — behandles automatisk som fejlleddet.
`VAR Settlement` navngiver den ene afhængige variabel.

`LABEL`-sætningen giver et mere læsevenligt visningsnavn for den
afhængige variabel i output-banneret. Outputtet producerer
koefficienterne for forventede middelkvadrater, variansanalysetabellen
(med en F-test af hver variankomponent mod nul), og estimatet af
**variankomponenten** på hvert niveau sammen med dens **procent af
totalen**.



In [3]:
TITEL 'Varianskomponenter i erstatningsudbetalinger for bilskader';
title2 'Tilfældige-effekter-ANOVA for Region / Filial / Sagsbehandler';

PROCEDURE nested data=claims;
   KLASSE Region Branch Adjuster;
   VARIABEL Settlement;
   MÆRKAT Region = 'Region' Branch = 'Filial' Adjuster = 'Sagsbehandler'
         Settlement = 'Erstatning udbetalt (USD)';
KØR;


                               Varianskomponenter i erstatningsudbetalinger for bilskader                               
                             Tilfældige-effekter-ANOVA for Region / Filial / Sagsbehandler                              


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Erstatning udbetalt (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Sagsbehandler  2313931.771806   272682.828942              8.9813      4.0000
Sagsbehandler         10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000


NOTE: Option TITLE changed to Varianskomponenter i erstatningsudbetalinger for bilskader.
NOTE: Option TITLE2 changed to Tilfældige-effekter-ANOVA for Region / Filial / Sagsbehandler.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Trin 4 — Analysér to afhængige variable på én gang

Ledelsen interesserer sig også for **sagsbehandlingstid** — hvor lang
tid skadessager tager at afregne. Vi tilføjer `CycleDays` til
`VAR`-listen. Med mere end én afhængig variabel rapporterer
`PROC NESTED` desuden en **kovariationsanalyse**, der viser, hvordan
de to afhængige variable co-varierer på hvert niveau af hierarkiet
(for eksempel om de regioner, der betaler mere, også afregner
langsommere).



In [4]:
TITEL 'Varianskomponenter for erstatningsbeløb og sagsbehandlingstid';
title2 'To afhængige variable i samme indlejrede hierarki';

PROCEDURE nested data=claims;
   KLASSE Region Branch Adjuster;
   VARIABEL Settlement CycleDays;
   MÆRKAT Region = 'Region' Branch = 'Filial' Adjuster = 'Sagsbehandler'
         Settlement = 'Erstatning udbetalt (USD)'
         CycleDays  = 'Dage til afregning';
KØR;


                             Varianskomponenter for erstatningsbeløb og sagsbehandlingstid                              
                                   To afhængige variable i samme indlejrede hierarki                                    


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Erstatning udbetalt (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Sagsbehandler  2313931.771806   272682.828942              8.9813      4.0000
Sagsbehandler         10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000


NOTE: Option TITLE changed to Varianskomponenter for erstatningsbeløb og sagsbehandlingstid.
NOTE: Option TITLE2 changed to To afhængige variable i samme indlejrede hierarki.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Trin 5 — Kun variansvisning med AOV-optionen

For et hurtigt overblik over variankomponenterne på tværs af begge
afhængige variable begrænser `AOV`-optionen outputtet til
variansanalysestatistikkerne og **undertrykker
kovariationsanalyse**-afsnittet. Dette er den kompakte visning, en
porteføljeanalytiker ville scanne, når vedkommende kun har brug for
variansfordelingen pr. niveau for hver afhængige variabel, ikke
co-variationen mellem dem.



In [5]:
TITEL 'Kun varianskomponenter (AOV)';

PROCEDURE nested data=claims aov;
   KLASSE Region Branch Adjuster;
   VARIABEL Settlement CycleDays;
   MÆRKAT Region = 'Region' Branch = 'Filial' Adjuster = 'Sagsbehandler'
         Settlement = 'Erstatning udbetalt (USD)'
         CycleDays  = 'Dage til afregning';
KØR;

TITEL;


                                              Kun varianskomponenter (AOV)                                              
                                   To afhængige variable i samme indlejrede hierarki                                    


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Erstatning udbetalt (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Filial  15528530.531717  1651824.844989             54.4057      8.0000
Filial                 5  11569658.859028          1.89      0.1829  Sagsbehandler  2313931.771806   272682.828942              8.9813      4.0000
Sagsbehandler         10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000


NOTE: Option TITLE changed to Kun varianskomponenter (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Fortolkning af resultaterne

Kolonnen **Percent of Total** i variansanalysetabellen er
hovedbudskabet. Læst top-til-bund giver den andelen af den samlede
afregningsvariabilitet, der kan tilskrives hvert organisatorisk lag.
For erstatningsbeløbet giver kørslen:

- **Region — 54.4%** (Pr > F = 0.0304). Dataene blev genereret med den
  største regionsspredning, og analysen genfinder den: over halvdelen
  af al variation i erstatningsbeløb er *mellem* regioner, statistisk
  signifikant evidens for, at *hvor* en skade behandles, er den
  dominerende driver.
- **Filial (inden for Region) — 9.0%** (Pr > F = 0.18). En beskeden
  ekstra andel, når man går fra region ned til den enkelte filial;
  ikke signifikant her.
- **Sagsbehandler (inden for Filial) — 3.7%** (Pr > F = 0.33). Kun
  lidt variation kan tilskrives den enkelte sagsbehandler i denne
  portefølje.
- **Fejlled — 32.9%.** Den resterende claim-til-claim-støj inden for
  en sagsbehandler; dette er den irreducible variation, som ingen
  organisatorisk håndtag kan fjerne.

Hvert niveau bærer også en **F-test (Pr > F)** af nulhypotesen om, at
dets variankomponent er nul. Den lave p-værdi for Region (0.0304) er
statistisk evidens for reelle systematiske forskelle mellem regioner,
ikke tilfældig stikprøvevariation.

Sagsbehandlingstiden fortæller en parallel historie: **Region står for
45.8%** af variationen i dage-til-afregning (Pr > F = 0.0448, igen
signifikant), med Filial og Sagsbehandler bidragende med
enkeltcifrede andele og residualet med 40.1%. Så både *hvor meget* et
forsikringsselskab betaler, og *hvor lang tid* det tager, styres
først og fremmest af regionen.

Kørslen med to afhængige variable tilføjer en
**kovariationsanalyse**. Her driver Region-niveauet krydsprodukterne,
og den samlede **korrelationskoefficient er -0.36** — en negativ
sammenhæng: de regioner, der betaler større erstatninger, har en
tendens til at *afregne dem hurtigere*, ikke langsommere. Det er et
nyttigt, ikke-oplagt fund — de dyre regioner er ikke de langsomme.

**Forretningskonklusion:** fordi Region dominerer procent-af-total-
fordelingen for begge afhængige variable, bør forsikringsselskabet
først standardisere afregningsretningslinjer og beløbsgrænser
*på tværs af regioner* — det er der, den største, statistisk
signifikante uensartethed findes — før der investeres i tiltag på
filial- eller sagsbehandlerniveau. Den negative sammenhæng mellem
erstatningsbeløb og sagsbehandlingstid betyder, at der ikke findes én
region, der både er den dyreste og den langsomste, så de to problemer
kan angribes med separate, regionsmålrettede programmer. PROC NESTED
gør en vag fornemmelse af, at "vores afregninger er uensartede", til
en handlingsorienteret, lag-for-lag-tilskrivning af, hvor den
uensartethed findes.

